# EXP1: Prompt-Category Separation — Colab runner

This notebook is a **thin caller**. It does not load a model, compute any
statistic, or contain any experiment logic -- all of that lives in
[`benchmarks/exp1_prompt_separation.py`](https://github.com/holeyfield33-art/unitarity-lab/blob/main/benchmarks/exp1_prompt_separation.py),
a committed module with its own argparse CLI.

The notebook only:
1. Installs the repo at a **pinned commit SHA**.
2. Runs the module as a subprocess.
3. Downloads the resulting JSON.

If a cell needs fixing to run in Colab, that fix belongs in the module (and
a new commit), not in this notebook -- a fix made only here would be
invisible and unversioned.

## 1. Pin the commit under test

Set `REF` to the exact commit you want to reproduce a number from. Everything
downstream is installed from that ref, not from a moving branch tip.

`"main"` is accepted for an exploratory run, but the resolved SHA is printed
and written into the manifest either way, so any number stays traceable to the
code that produced it.

In [ ]:
REF = "main"   # branch, tag, or full commit SHA
REPO_URL = "https://github.com/holeyfield33-art/unitarity-lab.git"

if "REPLACE" in REF:
    raise SystemExit(
        "REF is still a placeholder. Set it to a branch, tag, or commit SHA "
        "(e.g. \"main\") before running this cell."
    )

# Note the names: the repo is `unitarity-lab` (no trailing s) but the pip
# distribution is `unitarity-labs` (with s). `pip install unitarity-lab`
# fails -- that name is not on PyPI.
!pip install -q "unitarity-labs @ git+{REPO_URL}@{REF}"

import subprocess
resolved = subprocess.run(
    ["git", "ls-remote", REPO_URL, REF],
    capture_output=True, text=True,
).stdout.split()
resolved_sha = resolved[0] if resolved else REF
print(f"Installed unitarity-labs from {REPO_URL}@{REF}")
print(f"Resolved SHA: {resolved_sha}")

## 2. Run the experiment

All parameters below are passed straight through to
`benchmarks.exp1_prompt_separation`'s CLI -- see that module's `--help` for
the full flag list. No statistics or model logic run in this notebook.

In [ ]:
MODEL = "gpt2"
MODE = "passive"  # baseline | passive | active
PROMPTS_PER_CATEGORY = 8
MAX_NEW_TOKENS = 32
SEED = 42

!python -m benchmarks.exp1_prompt_separation \
    --model {MODEL} \
    --mode {MODE} \
    --prompts-per-category {PROMPTS_PER_CATEGORY} \
    --max-new-tokens {MAX_NEW_TOKENS} \
    --seed {SEED}

## 3. Locate and download the result JSON

The module writes `results/runs/<date>_<sha>_<env>/exp1_prompt_separation.json`
plus a `manifest.json` beside it. This cell only finds the most recent run
directory and hands the files to the browser -- it does not recompute or
reinterpret anything.

In [ ]:
import glob
import os

run_dirs = sorted(glob.glob("results/runs/*/"), key=os.path.getmtime)
assert run_dirs, "No run directory found -- did the module run successfully above?"
latest_run = run_dirs[-1]
print(f"Latest run: {latest_run}")

results_path = os.path.join(latest_run, "exp1_prompt_separation.json")
manifest_path = os.path.join(latest_run, "manifest.json")

try:
    from google.colab import files
    files.download(results_path)
    files.download(manifest_path)
except ImportError:
    print("Not running in Colab -- files are already on local disk at:")
    print(f"  {results_path}")
    print(f"  {manifest_path}")